# API FastAPI pour servir le modèle House Prices

Ce notebook démarre une API légère pour l'inférence. Le modèle est supposé enregistré dans `output/models/model_final.joblib` et produit des prédictions en dollars (après inverse de la transformée log1p).

## Pré-requis
- Installer les dépendances : `pip install fastapi uvicorn joblib pandas numpy`
- S'assurer que `output/models/model_final.joblib` existe (entraînement réalisé dans `feature_engineering.ipynb`).

## Endpoints
- `GET /health` : vérification de disponibilité
- `POST /predict` : prédictions batch, JSON avec liste d'objets `{"features": {col: valeur, ...}}`


In [1]:
# Cellule principale : API FastAPI + lancement uvicorn (depuis le notebook)
# Si uvicorn est déjà installé : commenter la ligne d'installation
# !pip install fastapi uvicorn joblib pandas numpy

import joblib
import numpy as np
import pandas as pd
from fastapi import FastAPI
from pydantic import BaseModel
from typing import Dict, Any, List
import nest_asyncio
import uvicorn

# Appliquer le patch pour ré-entrer la boucle event dans le notebook
nest_asyncio.apply()

MODEL_PATH = "output/models/model_final.joblib"

# Charger le modèle (pipeline complet avec préprocesseur si inclus)
model = joblib.load(MODEL_PATH)

app = FastAPI(title="HousePrice API", version="1.0.0")

class Item(BaseModel):
    features: Dict[str, Any]

@app.get("/health")
def health():
    return {"status": "ok"}

@app.post("/predict")
def predict(items: List[Item]):
    df = pd.DataFrame([it.features for it in items])
    y_log = model.predict(df)
    y = np.expm1(y_log)  # inverse de log1p -> prix en dollars
    return {"predictions": y.tolist()}

# Lancer l'API (port 8000 par défaut). Pour arrêter : Kernel → Interrupt
uvicorn.run(app, host="127.0.0.1", port=8000)


FileNotFoundError: [Errno 2] No such file or directory: 'output/models/model_final.joblib'